# Two-qubit crosstalk

Target: an ideal Rabi $\pi$-pulse on qubit $A$ only, lifted into the joint 2-qubit space via **embed**:
$H_{\text{target}}(t) = H_A(t) \otimes I_B$.

Realized: the same drive on $A$, plus a static residual $XX$ coupling to a spectator qubit $B$:
$H_{\text{real}}(t) = H_A(t) \otimes I_B + J\,\sigma_x^A \sigma_x^B$.

$B$ starts in $|0\rangle$ and is never driven -- any leakage into it comes purely from $J$.
This is the first demo needing **embed** (target $\to$ joint space) and **trace_out** (realized $\to$ qubit-$A$-only, via `core/subsystems.py`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from htdse.core.operator import Operator
from htdse.core.mechanism import Mechanism
from htdse.core.evolution import HamiltonianEvolution, UnitaryEvolution
from htdse.core.subsystems import embed
from htdse.submodules.spin import sigma_x, sigma_z, I2
from htdse.util import ket, otimes, process_fidelity, density_fidelity

DIMS = {"A": 2, "B": 2}

## Mechanisms

`RabiDrive` is qubit $A$'s ideal drive (same physics as the single-qubit demo). `EmbeddedTarget` wraps any
1-qubit mechanism with `embed` to produce the joint-space target. `Crosstalk` is the realized
2-qubit Hamiltonian with the $XX$ coupling to $B$.

In [ ]:
class RabiDrive(Mechanism):
    def __init__(self, Omega, eps=0.0, delta=0.0):
        self.Omega, self.eps, self.delta = Omega, eps, delta

    def hamiltonian(self, t) -> Operator:
        H = 0.5 * self.Omega * (1 + self.eps) * sigma_x + self.delta * sigma_z
        return Operator(H)


class EmbeddedTarget(Mechanism):
    def __init__(self, one_qubit_mechanism, dims, subsystem):
        self.mechanism, self.dims, self.subsystem = one_qubit_mechanism, dims, subsystem

    def hamiltonian(self, t) -> Operator:
        return embed(self.mechanism.hamiltonian(t), self.dims, self.subsystem)


class Crosstalk(Mechanism):
    def __init__(self, Omega, J, eps=0.0, delta=0.0):
        self.Omega, self.J, self.eps, self.delta = Omega, J, eps, delta

    def hamiltonian(self, t) -> Operator:
        H_A = 0.5 * self.Omega * (1 + self.eps) * sigma_x + self.delta * sigma_z
        return Operator(otimes(H_A, I2) + self.J * otimes(sigma_x, sigma_x))

## Gate-level comparison (embed, process fidelity)

Both propagators live on the full 4-dim joint space, so they're directly comparable -- no
reduction needed here, just the target lifted up via `embed`.

In [ ]:
Omega = 1.0
J = 0.08
T = np.pi / Omega

target = EmbeddedTarget(RabiDrive(Omega), DIMS, "A")
realized = Crosstalk(Omega, J=J)

U_target = UnitaryEvolution(target, dim=4).state_at(T)
U_real = UnitaryEvolution(realized, dim=4).state_at(T)
print(f"process fidelity: {process_fidelity(U_target, U_real):.6f}")

## State-level comparison (trace_out, density fidelity)

Evolve the actual 2-qubit state $|00\rangle$ under the realized Hamiltonian, then trace out $B$
and compare $\rho_A$ against the pure target outcome $|1\rangle_A$ via
$F(\rho_A, |1\rangle) = \langle 1|\rho_A|1\rangle$.

In [ ]:
psi0 = Operator(otimes(ket("0"), ket("0")))
evolution = HamiltonianEvolution(realized, psi0, subsystems=DIMS)
rho_A = evolution.trace_out("B", t=T)

purity = np.real(np.trace(rho_A @ rho_A))
print(f"density fidelity to |1>: {density_fidelity(rho_A, ket('1')):.6f}")
print(f"purity Tr(rho_A^2):      {purity:.6f}  (< 1 means A got entangled with B -- no dissipation, just leakage)")

## Purity vs. crosstalk strength

$\rho_A$ becomes mixed purely from entangling with $B$, with zero dissipation anywhere: mixedness
from a *modeled* finite subsystem is just `trace_out`, not Lindblad -- Lindblad is only needed when
the "bath" isn't part of the simulated Hilbert space at all (see `motional_dephasing.ipynb`).

In [ ]:
Js = np.linspace(0, 0.2, 15)
purities, fidelities = [], []
for j in Js:
    ev = HamiltonianEvolution(Crosstalk(Omega, J=j), psi0, subsystems=DIMS, verbose=False)
    rho = ev.trace_out("B", t=T)
    purities.append(np.real(np.trace(rho @ rho)))
    fidelities.append(density_fidelity(rho, ket("1")))

plt.plot(Js, purities, "o-", label=r"purity $\mathrm{Tr}(\rho_A^2)$")
plt.plot(Js, fidelities, "o-", label=r"$F(\rho_A, |1\rangle)$")
plt.xlabel("J")
plt.ylim(0, 1.05)
plt.legend()
plt.title("Qubit-A leakage vs. crosstalk strength")
plt.show()